In [0]:
from pyspark.sql.functions import col,concat_ws
from datetime import datetime


In [0]:
%run ./log_utils

In [0]:
# %sql
# INSERT INTO poc_catalog.default.config_table VALUES
# ('snowflake', 'SNOWFLAKE_SAMPLE_DATA', 'CUSTOMER', 'poc_catalog', 'snowflake', 'CUSTOMER', 'full', NULL, NULL, NULL)

In [0]:
# %sql
# DELETE FROM poc_catalog.config_schema.config_sf
# WHERE source_table = 'SF_TEST';


here

In [0]:
# %sql
# INSERT INTO poc_catalog.config_schema.config_sf VALUES
# ('snowflake', 'SNOWFLAKE_MIGRATION', 'SNOWFLAKE_MIGRATION_SCHEMA', 'SF_TEST', 'poc_catalog', 'snowflake', 'SF_TEST', 'incremental', 'CREATED_AT', "1970-01-01 00:00:00", "ID")

In [0]:
%sql
select * from poc_catalog.config_schema.config_sf

source,source_database,source_schema,source_table,target_table_catalog,target_table_schema,target_table_name,load_type,load_type_col_name,last_load_time,primary_key
snowflake,SNOWFLAKE_MIGRATION,SNOWFLAKE_MIGRATION_SCHEMA,SF_TEST,poc_catalog,snowflake,SF_TEST,incremental,CREATED_AT,1970-01-01T00:00:00.000Z,ID


In [0]:

logger = Logger()
jdbc_driver = "com.mysql.cj.jdbc.Driver"
scope_name = "AutoDbx"
sfURL = dbutils.secrets.get(scope_name, "sfURL")
sfUser = dbutils.secrets.get(scope_name, "sfUser")
sfPassword = dbutils.secrets.get(scope_name, "sfPassword")
# sfDatabase = "SNOWFLAKE_MIGRATION"
# sfSchema = "TPCH_SF1"
sfWarehouse = dbutils.secrets.get(scope_name, "sfWarehouse")
sfRole = dbutils.secrets.get(scope_name, "sfRole")

not running

In [0]:
def update_config_table(source_table_full, target_table_full):
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    spark.sql(f"""
        UPDATE poc_catalog.default.config_table
        SET last_load_time = TIMESTAMP('{current_time}')
        WHERE concat_ws('.', Source_Schema, Source_table) = '{source_table_full}'
          AND concat_ws('.',target_table_catalog ,Target_table_schema, Target_table_name) = '{target_table_full}'
    """)
    print(f"Updated config_table for {source_table_full} at {current_time}")

In [0]:
def get_last_load_time(source_table_full, target_table_full):
    result = spark.table("poc_catalog.config_schema.config_sf") \
                .filter(( concat_ws('.',col('source_schema'),col('source_table')) == source_table_full ) &
                        ( concat_ws('.',col('target_table_catalog'),col('target_table_schema'),col('target_table_name')) == target_table_full )) \
                .select("last_load_time").collect()
    return result[0]["last_load_time"] if result else None

In [0]:
def migrate_full_data(source_table_full, target_table_full):
    try:
        source_schema, source_table = source_table_full.split(".")
        target_catalog, target_schema, target_table = target_table_full.split(".")
        
        print(f"Starting full load: {source_table_full} -> {target_table_full}")
        df = spark.read \
              .format("snowflake") \
              .options(**sfOptions) \
              .option("dbtable", source_table) \
              .load()
        print(f"Full load successful: {source_table_full}")
        df.write.format("delta").mode("overwrite") \
          .saveAsTable(f"{target_catalog}.{target_schema}.{target_table}")

        update_config_table(source_table_full, target_table_full)
        logger.log_load(spark, "snowflake", source_table_full, target_table_full, "full", "success")
    except Exception as e:
        print(f"Full load failed: {source_table_full}")
        print(f"Error: {str(e)}")
        logger.log_load(spark, "snowflake", source_table_full, target_table_full, "full", "failed", str(e))
        logger.fetch_error(target_table_full, str(e))

In [0]:
def migrate_incremental_data(source_table_full, target_table_full,incremental_col_name, primary_key=None):
    try:
        source_schema, source_table = source_table_full.split(".")
        target_catalog, target_schema, target_table = target_table_full.split(".")
        primary_key = primary_key
        last_load_time = get_last_load_time(source_table_full, target_table_full)
        query = f"(SELECT * FROM {source_table} WHERE {incremental_col_name} > '{last_load_time}') AS src"

        print(f"Starting incremental load: {source_table_full} -> {target_table_full} after {last_load_time}")
        df = spark.read \
              .format("snowflake") \
              .options(**sfOptions) \
              .option("query", query) \
              .load()

        df.createOrReplaceTempView("incremental_view")

        spark.sql(f"""
            MERGE INTO {target_catalog}.{target_schema}.{target_table} AS target
            USING incremental_view AS source
            ON target.{primary_key} = source.{primary_key}
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

        print(f"Incremental load successful: {source_table_full}")
        update_config_table(source_table_full, target_table_full)
        logger.log_load(spark, "snowflake", source_table_full, target_table_full, "incremental", "success")
    except Exception as e:
        print(f"Incremental load failed for {source_table_full}: {e}")
        logger.log_load(spark, "snowflake", source_table_full, target_table_full, "incremental", "failed", str(e))
        logger.fetch_error(target_table_full, str(e)) 

In [0]:
%sql
SELECT * FROM poc_catalog.config_schema.config_sf where source = "snowflake"

source,source_database,source_schema,source_table,target_table_catalog,target_table_schema,target_table_name,load_type,load_type_col_name,last_load_time,primary_key
snowflake,SNOWFLAKE_MIGRATION,SNOWFLAKE_MIGRATION_SCHEMA,SF_TEST,poc_catalog,snowflake,SF_TEST,incremental,CREATED_AT,1970-01-01T00:00:00.000Z,ID


In [0]:
config_df = spark.table("poc_catalog.config_schema.config_sf") \
              .filter(col("source") == "snowflake") \
              .select("source_schema", "source_database", "source_table", "target_table_catalog","target_table_schema","target_table_name", "load_type","load_type_col_name", "primary_key")

for row in config_df.collect():
    source_table = f"{row['source_schema']}.{row['source_table']}"
    target_table = f"{row['target_table_catalog']}.{row['target_table_schema']}.{row['target_table_name']}"
    incremental_col_name = row["load_type_col_name"]
    load_type = row["load_type"]
    primary_key = row["primary_key"]
    sfDatabase = row['source_database']
    sfSchema = row['source_schema']

    sfOptions = {
    "sfURL": sfURL,
    "sfUser": sfUser,                           
    "sfPassword": sfPassword,                     
    "sfDatabase": sfDatabase,               
    "sfSchema": sfSchema,                              
    "sfWarehouse": sfWarehouse,                         
    "sfRole": sfRole                             
    }
    
    if load_type == "full":
        migrate_full_data(source_table, target_table)
    elif load_type == "incremental":
        migrate_incremental_data(source_table, target_table,incremental_col_name, primary_key)
        # migrate_full_data(source_table, target_table)
    else:
        print(f"Unknown load_type for {source_table}")
        logger.log_load(spark, "mysql", source_table, target_table, load_type, "failed", "Unknown load_type")
        logger.fetch_error(target_table, "Unknown load_type")

Starting incremental load: SNOWFLAKE_MIGRATION_SCHEMA.SF_TEST -> poc_catalog.snowflake.SF_TEST after 1970-01-01 00:00:00
Incremental load successful: SNOWFLAKE_MIGRATION_SCHEMA.SF_TEST
Updated config_table for SNOWFLAKE_MIGRATION_SCHEMA.SF_TEST at 2025-06-20 10:49:15


In [0]:
errors = logger.return_error()
if len(errors) > 0:
    raise Exception("Error occurred during migration")